# 02 - Build the Previous Growth / Safe / Risky Master with PostgreSQL

            The original master dataset was constructed entirely in PostgreSQL. Python in this notebook only loads the five raw CSV files, executes `02_OLD_MASTER_PIPELINE.sql`, exports the final table and compares it with the frozen master. No financial feature or label is calculated in Pandas.


In [ ]:
from pathlib import Path

def find_package_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "01_DATA_PIPELINE").is_dir() and (candidate / "02_MODELS").is_dir():
            return candidate
    if current.name == "01_DATA_PIPELINE":
        return current.parent
    raise RuntimeError("Open this notebook from inside the SUBMIT_THIS package.")

PACKAGE_ROOT = find_package_root()
PIPELINE_DIR = PACKAGE_ROOT / "01_DATA_PIPELINE"
FROZEN_RAW_DIR = PIPELINE_DIR / "FROZEN_RAW_DATA"
GENERATED_DIR = PIPELINE_DIR / "GENERATED_OUTPUTS"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print("Package root:", PACKAGE_ROOT)


In [ ]:
import getpass
import importlib.util
import os
import subprocess
import sys

DATA_SOURCE = "frozen"  # Change to "live" only after notebook 01 finishes successfully.
DB_HOST = os.environ.get("PGHOST", "localhost")
DB_PORT = int(os.environ.get("PGPORT", "5432"))
DB_NAME = "bank_stock_ml_submit"
DB_USER = os.environ.get("PGUSER", "postgres")
DB_PASSWORD = os.environ.get("PGPASSWORD")
if DB_PASSWORD is None:
    DB_PASSWORD = getpass.getpass("PostgreSQL password: ")

VENDOR_DIR = PIPELINE_DIR / "_vendor"
VENDOR_DIR.mkdir(parents=True, exist_ok=True)
if str(VENDOR_DIR) not in sys.path:
    sys.path.insert(0, str(VENDOR_DIR))
if importlib.util.find_spec("psycopg") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--target", str(VENDOR_DIR), "psycopg[binary]"
    ])

if DATA_SOURCE == "frozen":
    RAW_DIR = FROZEN_RAW_DIR
else:
    RAW_DIR = GENERATED_DIR / "01_RAW_LIVE" / "data"

OUTPUT_DIR = GENERATED_DIR / "02_OLD_MASTER_SQL"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SQL_PATH = PIPELINE_DIR / "02_OLD_MASTER_PIPELINE.sql"
FROZEN_MASTER = PACKAGE_ROOT / "02_MODELS" / "01_DECISION_TREE_GROWTH_SAFE_RISK" / "master_ml_training.csv"

print("Raw source:", RAW_DIR)
print("Dedicated database:", DB_NAME)
print("Output:", OUTPUT_DIR)


In [ ]:
import pandas as pd
import psycopg
from psycopg import sql

raw_files = {
    "company_raw": RAW_DIR / "company_raw.csv",
    "daily_market_raw": RAW_DIR / "daily_market_raw.csv",
    "quarterly_fundamental_raw": RAW_DIR / "quarterly_fundamental_raw.csv",
    "quarterly_reportnorm_raw_long": RAW_DIR / "quarterly_reportnorm_raw_long.csv",
    "annual_ocr_raw": RAW_DIR / "annual_ocr_raw.csv",
}
for path in raw_files.values():
    if not path.is_file():
        raise FileNotFoundError(path)

admin = psycopg.connect(host=DB_HOST, port=DB_PORT, dbname="postgres", user=DB_USER, password=DB_PASSWORD, autocommit=True)
with admin.cursor() as cursor:
    cursor.execute("select 1 from pg_database where datname = %s", (DB_NAME,))
    if cursor.fetchone() is None:
        cursor.execute(sql.SQL("create database {}").format(sql.Identifier(DB_NAME)))
        print("Created database", DB_NAME)
admin.close()

connection = psycopg.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
connection.execute("create schema if not exists raw")
connection.commit()

def load_csv_as_text(table_name: str, path: Path) -> int:
    header = list(pd.read_csv(path, nrows=0, encoding="utf-8-sig").columns)
    with connection.cursor() as cursor:
        cursor.execute(sql.SQL("drop table if exists raw.{} cascade").format(sql.Identifier(table_name)))
        columns_sql = sql.SQL(", ").join(
            sql.SQL("{} text").format(sql.Identifier(column)) for column in header
        )
        cursor.execute(
            sql.SQL("create table raw.{} (source_row_number bigserial primary key, {})").format(
                sql.Identifier(table_name), columns_sql
            )
        )
        copy_sql = sql.SQL("copy raw.{} ({}) from stdin with (format csv, header true)").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(sql.Identifier(column) for column in header),
        )
        with cursor.copy(copy_sql) as copy:
            with path.open("r", encoding="utf-8-sig", newline="") as handle:
                while chunk := handle.read(1024 * 1024):
                    copy.write(chunk)
        cursor.execute(sql.SQL("select count(*) from raw.{}").format(sql.Identifier(table_name)))
        count = cursor.fetchone()[0]
    connection.commit()
    return count

load_summary = []
for table, path in raw_files.items():
    count = load_csv_as_text(table, path)
    load_summary.append({"table": f"raw.{table}", "rows": count})
display(pd.DataFrame(load_summary))


In [ ]:
sql_text = SQL_PATH.read_text(encoding="utf-8")
with connection.cursor() as cursor:
    cursor.execute(sql_text, prepare=False)
connection.commit()
print("PostgreSQL transformation pipeline completed.")


In [ ]:
output_csv = OUTPUT_DIR / "master_ml_training.csv"
with connection.cursor() as cursor:
    with cursor.copy(
        "copy (select * from final.master_ml_training_export order by period_end, ticker) "
        "to stdout with (format csv, header true, encoding 'UTF8')"
    ) as copy:
        with output_csv.open("wb") as handle:
            for block in copy:
                handle.write(block)

acceptance = pd.read_sql("select * from audit.old_master_acceptance order by check_name", connection)
display(acceptance)
connection.close()

generated = pd.read_csv(output_csv)
frozen = pd.read_csv(FROZEN_MASTER)
schema_match = list(generated.columns) == list(frozen.columns)
key_match = set(zip(generated.ticker, generated.period_end)) == set(zip(frozen.ticker, frozen.period_end))
label_check = generated[["ticker", "period_end", "segment_label"]].merge(
    frozen[["ticker", "period_end", "segment_label"]],
    on=["ticker", "period_end"], suffixes=("_generated", "_frozen")
)
labels_match = bool((label_check.segment_label_generated == label_check.segment_label_frozen).all())

result = {
    "rows": len(generated),
    "columns": generated.shape[1],
    "schema_matches_frozen": schema_match,
    "business_keys_match_frozen": key_match,
    "labels_match_frozen": labels_match,
}

merged = generated.merge(frozen, on=["ticker", "period_end"], suffixes=("_generated", "_frozen"))
numeric_differences = []
for column in generated.columns:
    if column in {"ticker", "period_end", "segment_label", "bank_quarter_id"}:
        continue
    generated_name = column + "_generated"
    frozen_name = column + "_frozen"
    if generated_name not in merged or frozen_name not in merged:
        continue
    left = pd.to_numeric(merged[generated_name], errors="coerce")
    right = pd.to_numeric(merged[frozen_name], errors="coerce")
    both = left.notna() & right.notna()
    max_difference = float((left[both] - right[both]).abs().max()) if both.any() else 0.0
    mismatch_rows = int(((left - right).abs() > 1e-6).fillna(False).sum())
    null_mismatch_rows = int((left.isna() != right.isna()).sum())
    if mismatch_rows or null_mismatch_rows:
        numeric_differences.append({
            "column": column,
            "mismatch_rows": mismatch_rows,
            "null_mismatch_rows": null_mismatch_rows,
            "max_abs_difference": max_difference,
        })

result["numeric_columns_with_material_difference"] = len(numeric_differences)
result["numeric_difference_detail"] = numeric_differences
display(pd.DataFrame([result]))
(OUTPUT_DIR / "NOTEBOOK_ACCEPTANCE.json").write_text(
    __import__("json").dumps(result, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Exported:", output_csv)
